# Profiler trace analysis

The `train.py` script containing the solution is available as [`solution/train.py`](solution/train.py).

This notebook analyses the `torch.profiler` traces produced by `train.py` using
[Holistic Trace Analysis (HTA)](https://github.com/facebookresearch/HolisticTraceAnalysis).
It replaces the deprecated `torch-tb-profiler` TensorBoard plugin.

**Prerequisites**
```bash
pip install HolisticTraceAnalysis
```

**Trace location** — copy traces from the cluster first:
```bash
rsync -r <host>:~/<path>/4_profile/profiler_output_kernels ./
```

In [ ]:
import glob, os
from hta.trace_analysis import TraceAnalysis
import plotly.offline as pyo
pyo.init_notebook_mode() # Set notebook mode to work in offline

TRACE_DIR_A = os.path.abspath("./profiler_output_kernels")  # Run A: accurate kernel timings


def load_trace_files(trace_dir: str) -> dict[int, str]:
    """Return a Dict[rank, absolute_path] for all .pt.trace.json files under trace_dir."""
    files = sorted(glob.glob(os.path.join(trace_dir, "**", "*.pt.trace.json"), recursive=True))
    if not files:
        raise FileNotFoundError(
            f"No .pt.trace.json files found under {trace_dir!r}.\n"
            "Copy traces from the cluster first:\n"
            "  rsync -r <host>:~/<path>/4_profile/profiler_output_kernels ./"
        )
    result = {}
    for fp in files:
        rank_dir = os.path.basename(os.path.dirname(fp))  # e.g. 'rank_0'
        rank = int(rank_dir.split("_")[-1]) if rank_dir.startswith("rank_") else len(result)
        result[rank] = fp
    return result


trace_files_a = load_trace_files(TRACE_DIR_A)

print("Run A — kernel timing traces:")
for rank, fp in sorted(trace_files_a.items()):
    print(f"  rank {rank}: {fp}")

analyzer_a = TraceAnalysis(trace_files=trace_files_a)

---
## 1. Overview - Temporal breakdown

`get_temporal_breakdown()` splits total profiling window into three buckets:

| Column | Meaning |
|---|---|
| `idle_time` | GPU had **no kernel running** — waiting for the CPU to launch the next op |
| `compute_time` | GPU was running a **compute kernel** (matrix multiply, elementwise, etc.) |
| `non_compute_time` | GPU was running a **non-compute kernel** (memory copy, NCCL collective, etc.) |
| `kernel_time` | `compute_time + non_compute_time` — any kernel was running |
| `compute_time_pctg` | `compute_time / (kernel_time + idle_time) × 100` — fraction of total time doing compute |
| `idle_time_pctg` | `idle_time / (kernel_time + idle_time) × 100` — fraction of total time the GPU was idle |

**GPU utilisation** ≈ `100 - idle_time_pctg`. A high idle fraction means the GPU is waiting — either for the CPU to launch kernels or for a collective to complete.

In [ ]:
temporal_breakdown = analyzer_a.get_temporal_breakdown(visualize=True)
temporal_breakdown

### Per-step breakdown

`get_temporal_breakdown()` aggregates over the whole trace. To see how each `ProfilerStep`
compares, we access the raw trace DataFrame directly — each GPU kernel event already has an
`iteration` column set by HTA during loading.

| Column | Meaning |
|---|---|
| `step` | `ProfilerStep` number from the profiler schedule (e.g. 3, 4, 5 for `active=3`) |
| `span_ms` | Wall time from the **first kernel start** to the **last kernel end** in this step |
| `active_ms` | Total GPU time with at least one kernel running (overlapping kernels counted once) |
| `idle_ms` | `span - active` — gaps between kernels where the GPU was waiting for the CPU |
| `gpu_util_pct` | `active / span × 100` — fraction of the step the GPU was doing useful work |

Steps with consistent `span_ms` and `gpu_util_pct` confirm stable, repeatable training iterations.
A step with lower `gpu_util_pct` than the others suggests CPU stalls or a synchronisation point.

In [ ]:
import pandas as pd

def merged_active_time(df: pd.DataFrame) -> int:
    """Total us covered by GPU kernels after merging overlapping intervals."""
    intervals = df[["ts", "end"]].sort_values("ts").values.tolist()
    merged = []
    for start, end in intervals:
        if merged and start <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], end)
        else:
            merged.append([start, end])
    return sum(e - s for s, e in merged)


rows = []
for rank, df in analyzer_a.t.traces.items():
    gpu = df[df["stream"].gt(0)].copy()
    gpu["end"] = gpu["ts"] + gpu["dur"]

    for step, step_df in gpu.groupby("iteration"):
        if step == -1:          # events not attributed to any ProfilerStep
            continue
        span        = step_df["end"].max() - step_df["ts"].min()
        active      = merged_active_time(step_df)
        idle        = span - active
        rows.append({
            "rank":             rank,
            "step":             step,
            "span_ms":          round(span   / 1000, 1),
            "active_ms":        round(active / 1000, 1),
            "idle_ms":          round(idle   / 1000, 1),
            "gpu_util_pct":     round(active / span * 100, 1),
        })

per_step = pd.DataFrame(rows).sort_values(["rank", "step"])
per_step

### Idle time breakdown

Drills into **why** the GPU was idle. The total idle time is split into three categories per CUDA stream:

| Category | Meaning |
|---|---|
| **Host wait** | The CPU has not enqueued enough kernels to keep the stream busy — the GPU is starved. Common causes: slow data loading, Python overhead, or host-to-device copies blocking the CPU thread. |
| **Kernel wait** | Short gaps between consecutive kernels (< 30 ns by default) attributed to the overhead of back-to-back kernel launches. Expected in any well-pipelined workload. |
| **Other wait** | Idle time from an unknown cause — typically a compute stream waiting for a CUDA event from a communication stream (e.g. NCCL allreduce) to complete before proceeding. |

**What to look for:**
- High **host wait** → the CPU is the bottleneck; profile the data pipeline or reduce Python overhead.
- High **other wait** → a cross-stream dependency is on the critical path; check the Perfetto trace to identify which event the stream is waiting on.
- **Kernel wait** should be small and roughly proportional to the number of kernel launches — if it is large, you may have too many tiny kernels that could be fused.

In [ ]:
idle_time_df = analyzer_a.get_idle_time_breakdown()

### Questions to consider

<details>
<summary><b>1. What fraction of the profiling window was the GPU idle? Is it above 5%?</b></summary>

Check `idle_time_pctg` in the aggregated breakdown. Values below ~2% indicate the GPU is continuously fed work — the CPU is keeping up with kernel launches. Above 5% suggests the CPU is a bottleneck (slow data loading, Python overhead, or host-to-device copies stalling the pipeline). See Section 5 (CUDA kernel launch stats) to identify which CPU operations are causing the gaps.
</details>

<details>
<summary><b>2. Does compute dominate over non-compute?</b></summary>

Compare `compute_time_pctg` vs `non_compute_time_pctg`. In a well-optimised training run, compute (matrix multiplies, activations) should dominate. A large `non_compute_time_pctg` means the GPU is spending significant time on memory copies (`aten::copy_` for host-to-device) or NCCL collectives. For multi-GPU runs, check whether NCCL kernels overlap with backward compute (communication hiding). See Section 2 (GPU kernel breakdown) to identify the specific kernels.
</details>

<details>
<summary><b>3. Are all steps consistent in <code>span_ms</code> and <code>gpu_util_pct</code>?</b></summary>

In the per-step table, all active steps should have similar `span_ms`. An outlier (usually the first active step) means JIT compilation or allocator warm-up slipped past the profiler's `warmup` phase — consider increasing `warmup` in the schedule. A drop in `gpu_util_pct` for a specific step points to a synchronisation barrier or a data-loading stall at that iteration.
</details>

<details>
<summary><b>4. Does <code>idle_ms</code> grow with step number?</b></summary>

Increasing `idle_ms` across steps can indicate memory pressure: the allocator fragments over time, triggering garbage collection pauses that stall CPU-to-GPU launches. If `span_ms` grows but `active_ms` stays flat, this is the likely cause. Check peak memory usage in Section 4 to see whether you are close to the VRAM limit.
</details>

<details>
<summary><b>5. (Multi-GPU) Is non-compute time significant? Is NCCL overlapping backward?</b></summary>

On multi-GPU runs, `non_compute_time_pctg` captures NCCL allreduce kernels. Ideally these overlap with the backward pass (gradient bucketing in DDP). If `non_compute_time` is large and does **not** overlap backward compute, communication is on the critical path. Use the Perfetto trace view (Section 4) to check whether `ncclKernel_*` events run concurrently with `gemm`/`ampere_*` events on separate CUDA streams.
</details>

---
## 2. GPU Kernel — kernel time breakdown

Section 1 told you *how much* time was non-compute. Section 2 tells you *what* that time was spent on and *which specific kernels* caused it.

`get_gpu_kernel_breakdown()` classifies every GPU kernel into one of three types:

| HTA kernel type | Maps to Section 1 bucket | Meaning |
|---|---|---|
| `COMPUTATION` | `compute_time` | Matrix multiplies, activations, layer norms — useful work |
| `COMMUNICATION` | `non_compute_time` | NCCL collectives (AllReduce, AllGather) — inter-GPU communication |
| `MEMORY` | `non_compute_time` | Device-to-device copies, memset — data movement within the GPU |

So `non_compute_time_pctg` from Section 1 ≈ `COMMUNICATION% + MEMORY%` from Table 1 below. If `non_compute_time_pctg` was high:
- a large `COMMUNICATION` share → NCCL collectives are on the critical path (not hidden behind compute)
- a large `MEMORY` share → unexpected tensor copies are happening (check for `aten::copy_` or H2D transfers)

Table 2 then gives you the specific kernel names, so you can pinpoint which `ncclKernel_*` or copy kernel is responsible.

---

In [ ]:
kernel_type_metrics_df, kernel_metrics_df = analyzer_a.get_gpu_kernel_breakdown(visualize=False)

In [ ]:
kernel_type_metrics_df

### Table 1 — `kernel_type_metrics_df` (kernel type summary)

One row per kernel type, aggregated across all ranks.

| Column | Meaning |
|---|---|
| `kernel_type` | `COMPUTATION`, `COMMUNICATION`, or `MEMORY` |
| `sum` | Total µs spent in kernels of this type (merged intervals, no double-counting of overlap) |
| `percentage` | Fraction of total GPU kernel time (0–1) |
---

In [ ]:
kernel_metrics_df

### Table 2 — `kernel_metrics_df` (per-kernel statistics)

One row per unique kernel name × rank. Shows the top kernels that together account for `duration_ratio` (default 80%) of their kernel type's total time, capped at `num_kernels` (default 10).

| Column | Meaning |
|---|---|
| `name` | CUDA kernel name (shortened) |
| `kernel_type` | `COMPUTATION`, `COMMUNICATION`, or `MEMORY` |
| `rank` | GPU rank |
| `sum (us)` | Total µs this kernel ran across all profiled steps |
| `mean (us)` | Average duration per launch |
| `min (us)` / `max (us)` | Fastest and slowest individual launch |
| `stddev` | Standard deviation of launch duration — high values indicate variable kernel performance |

### How to read the results

**Kernel names are the key signal.** Look at the `name` column in `kernel_metrics_df`:

| Pattern in name | Meaning |
|---|---|
| `simt_sgemm` | FP32 matrix multiply on regular CUDA cores — **Tensor Cores idle** |
| `tensorop` / `ampere_` / `xmma` | Tensor Core kernel — active for FP16/BF16 or TF32 |
| `fmha_cutlass*_f32` | Flash Attention in FP32 — becomes `_bf16` after `model.to(torch.bfloat16)` |
| `ncclKernel_*` | NCCL collective — communication on the critical path if large |
| `vectorized_elementwise` | Elementwise op (activation, dropout) — small, expected |
| `reduce_kernel` | Reduction (layer norm, loss) — small, expected |

**If you see `simt_sgemm` dominating** (as in the FP32 baseline): Tensor Cores are completely idle. Switching to BF16 (`model.to(torch.bfloat16)`) replaces these with `tensorop` / `ampere_` kernels and is typically the single highest-impact optimisation available.

**`stddev = 0`** for a kernel means it ran exactly once across all profiled steps — normal for a fixed-shape model. High stddev on a compute kernel suggests variable input shapes or memory pressure.

> **Important: `stddev = 0` can be a profiler artefact, not a real signal**
>
> Features like DDP gradient bucketing, `torch.compile`, and CUDA Graphs introduce synchronisation overheads or abstraction layers that cause the PyTorch Profiler to report distorted or single-instance metrics. As a result, HTA will show `stddev = 0` on kernels that actually run many times (because they are fused or captured into a single graph node), and execution counts and averaged timings will be inaccurate.
>
> To get reliable per-kernel statistics you have two options:
> - **Disable these features** for the profiling run (re-enable afterwards): comment out `torch.compile()`, set `bucket_cap_mb=0` in DDP, and avoid CUDA Graph capture during the profiled steps.
> - **Use NVIDIA Nsight Systems instead** (Exercise 5) — `nsys` profiles at the driver level, sees through all PyTorch abstractions, and reports accurate execution counts and durations regardless of whether `torch.compile` or CUDA Graphs are active.

### Questions to consider

<details>
<summary><b>1. Does COMPUTATION dominate kernel time?</b></summary>

Check `percentage` in `kernel_type_metrics_df`. In a well-optimised single-GPU run, `COMPUTATION` should account for >90% of kernel time. If `COMMUNICATION` or `MEMORY` is significant, the GPU is spending time on work that is not directly producing outputs — see questions 2 and 3.
</details>

<details>
<summary><b>2. Are Tensor Cores active?</b></summary>

Scan the `name` column in `kernel_metrics_df` for the top COMPUTATION kernels. Names containing `simt_sgemm` indicate FP32 matrix multiplies on regular CUDA cores — Tensor Cores are idle. Names containing `tensorop`, `ampere_`, or `xmma` indicate Tensor Core kernels (FP16/BF16/TF32). If `simt_sgemm` dominates, switching to BF16 (`model.to(torch.bfloat16)`) is the single highest-impact optimisation available.
</details>

<details>
<summary><b>3. Is COMMUNICATION time significant, and is it hidden?</b></summary>

On multi-GPU runs, check whether `COMMUNICATION` percentage is large. If so, look at whether NCCL kernels (`ncclKernel_*`) overlap with backward compute in the Trace view (Section 4). Non-overlapping NCCL means communication is on the critical path — consider using `torch.compile`, `gradient_as_bucket_view=True`, or `find_unused_parameters=False` in DDP.
</details>

<details>
<summary><b>4. Is any single kernel dominating unexpectedly?</b></summary>

Look at `sum (us)` in `kernel_metrics_df`. If one kernel accounts for a disproportionate share of its type's total time, it is worth investigating. High `stddev` on a compute kernel suggests variable input shapes or memory pressure causing inconsistent execution time. `stddev = 0` simply means the kernel ran exactly once across all profiled steps.
</details>

> **Important: the "heavyweight trinity" distorts GPU kernel timings**
>
> If the profiler in `train.py` is configured with all three of these flags enabled simultaneously:
> ```python
> record_shapes=True
> with_stack=True
> profile_memory=True
> ```
> Enabling all three at once introduces **extremely severe overhead**. This setup is invaluable for debugging — it tells you which line of code generated a specific tensor, where memory leaks occur, and what shapes flow through each op — but it **completely destroys the accuracy of GPU kernel time measurements**.
>
> When `profile_memory=True` and `with_stack=True` are both active, PyTorch forces the CPU to do a large amount of extra work before and after **every single GPU operation**: capturing the Python call stack and recording allocator events. This additional CPU activity inflates per-kernel timings and can even alter which kernels appear on the critical path.
>
> **To measure accurate GPU kernel execution speed**, disable these flags:
> ```python
> record_shapes=False,
> with_stack=False,
> profile_memory=False,
> ```
> Alternatively, use **NVIDIA Nsight Systems** (Exercise 5), which profiles at the driver level and adds no measurable overhead to GPU kernels.

---
## 3. Memory — bandwidth

**How fast data is moving** (memcpy/memset bandwidth). Check bandwidth counters for unexpected data movement between host and device or within device memory.

> **Memory allocation timeline** (peak allocated/reserved, per-step spike pattern) requires a separate profiling run with `profile_memory=True`. See `analysis_mem_usage.ipynb` and Part B in the README.

### 3a. Bandwidth time series

`get_memory_bw_time_series()` returns a time series of memcpy/memset bandwidth sampled over the profiling window. Each row is one time bucket; columns are H2D, D2H, and D2D bandwidth in GB/s. Useful for spotting bursts of data transfer (e.g. a data-loader copy that spikes at the start of each step).

In [ ]:
mem_bw_ts = analyzer_a.get_memory_bw_time_series()
mem_bw_ts

### 3b. Bandwidth summary

Aggregate statistics for each copy type (H2D, D2H, D2D). Columns are count, min, max, mean, stddev, and percentiles of individual operation bandwidth in GB/s. Compare D2D peak to the theoretical maximum for your GPU (e.g. ~2 TB/s for A100 SXM, ~3.35 TB/s for GH200).

In [ ]:
mem_summary = analyzer_a.get_memory_bw_summary()
mem_summary

### Questions to consider

<details>
<summary><b>1. What do near-zero H2D / D2H bandwidth values mean?</b></summary>

Sections 3a and 3b measure *explicit memory copy operations* (`cudaMemcpy`) — data moving between CPU (Host) and GPU (Device), or between GPU memory regions (Device-to-Device). They do **not** measure bandwidth used by compute kernels reading their own buffers; that is internal to the GPU.

Near-zero H2D and D2H simply confirm the profiled steps had no CPU→GPU transfers. This is expected when the input batch was already on device before training started. Large H2D spikes during training would indicate the data loader is not pre-fetching to GPU, causing blocking transfers each step.
</details>

<details>
<summary><b>2. Is D2D bandwidth close to the theoretical peak?</b></summary>

Compare the D2D max from `get_memory_bw_summary()` against your GPU's rated HBM bandwidth (e.g. ~3.35 TB/s for GH200). A large gap may indicate small, non-coalesced copies or memory pressure from fragmentation.
</details>

---
## 4. Trace view — Perfetto UI

HTA does not replace the interactive timeline. Use **Perfetto UI** instead:

1. Open [ui.perfetto.dev](https://ui.perfetto.dev) in Chrome or Firefox.
2. Drag and drop a `.pt.trace.json` file from `profiler_output_kernels/rank_0/`.
3. Zoom into a `ProfilerStep` block to see CPU threads and GPU streams on the same axis.
4. Click any event to read kernel duration, stream ID, and (with `with_stack=True`) the
   Python call stack that triggered it.

**Things to look for in the trace:**
- Gaps between consecutive CUDA kernels → GPU starvation
- Long `ncclAllReduce` blocks overlapping (or not overlapping) backward kernels → communication hiding
- `cudaMemcpy` calls between compute kernels → unexpected data movement

The cell below prints the path to the trace file for convenience.

---
## 5. Operator — CUDA kernel launch statistics

Links each GPU kernel to its paired CPU dispatch event via `correlation_id` and produces three graphs. Use Run A (all flags off) for this section — `profile_memory=True` inflates `cpu_duration` and floods Graph 1 with false positives.

| Graph | Signal | Act when… |
|---|---|---|
| **1. Short GPU kernels** (`gpu_duration < cpu_duration`) | CPU took longer to dispatch than the GPU took to run — a guaranteed stall | Many such kernels appear → ops are too small to keep the GPU fed; fuse with `torch.compile` or replace Python loops with batched ops |
| **2. Long runtime calls** (`cpu_duration > 50 µs`) | CUDA driver was blocked during `cudaLaunchKernel` — starves subsequent launches | Spikes appear → check for unnecessary `torch.cuda.synchronize()`, in-path `cudaMalloc`, or DDP barriers serialising steps |
| **3. Launch delay outliers** (`launch_delay > 100 µs`) | GPU had to wait after CPU finished dispatching — stream was backlogged | Large delay on a *small* kernel after a sync point → unexpected serialisation; large delay behind a `gemm` is normal and expected |

**On a well-utilised GPU (idle < 2%, compute-dominant)** all three graphs will be sparse — Section 5 adds little beyond confirming the CPU is keeping up. It becomes most valuable after applying optimisations (`torch.compile`, BF16, CUDA Graphs) to check whether they introduced new CPU stalls or unexpected synchronisations.

In [ ]:
launch_stats = analyzer_a.get_cuda_kernel_launch_stats(visualize=True)
launch_stats